In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

TRAIN_PATH = "train.csv"
TEST_PATH = "test-unlabelled.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Columnas finales:", train.columns[-5:].tolist())

X = train.drop(columns=["class"])
y = train["class"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

def evaluate_model(name, model, notes=""):
    """
    Evalúa un modelo con:
    - accuracy en todo el training set
    - accuracy media con 5-fold cross-validation
    """
    model.fit(X, y)
    train_pred = model.predict(X)
    train_acc = accuracy_score(y, train_pred)

    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    row = {
        "experiment": name,
        "train_accuracy": train_acc,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "notes": notes
    }
    results.append(row)

    print(f"Experimento: {name}")
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"5-fold CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Notas: {notes}")

    return model, cv_scores

Train shape: (1000, 610)
Test shape: (2000, 610)
Columnas finales: ['nwpqkzh', 'yjsknnh', 'qtfvayr', 'kaxienf', 'class']


In [2]:
print("Dimensiones X:", X.shape)
print("\nDistribución de clases:")
print(y.value_counts())
print("\nProporción de clases:")
print(y.value_counts(normalize=True))

print("\nValores nulos en train:", train.isna().sum().sum())
print("Valores nulos en test:", test.isna().sum().sum())

X.describe().T.head()

Dimensiones X: (1000, 609)

Distribución de clases:
class
A    347
B    328
C    325
Name: count, dtype: int64

Proporción de clases:
class
A    0.347
B    0.328
C    0.325
Name: proportion, dtype: float64

Valores nulos en train: 0
Valores nulos en test: 2000


,count,mean,std,min,25%,50%,75%,max
lnauxax,1000.0,6.962000,2.654355,0.000000,5.000000,7.000000,9.000000,16.000000
frpcvrs,1000.0,3.085164,2.429008,0.025600,1.332050,2.510294,4.110204,16.984291
vhztoil,1000.0,4.888311,3.110253,0.216418,2.583480,4.204823,6.614121,19.568443
jvvthfj,1000.0,-0.407401,8.232834,-109.570960,-2.187783,0.787451,2.583759,38.072782
oniwlqr,1000.0,3.015783,2.457959,0.010424,1.207115,2.347860,4.295479,20.342250


## Gaussian Naive Bayes baseline

In [3]:
from sklearn.naive_bayes import GaussianNB

nb_baseline = GaussianNB()

model_nb_baseline, scores_nb_baseline = evaluate_model(
    name="NB-01 GaussianNB baseline",
    model=nb_baseline,
    notes="Baseline sin preprocesamiento. Sirve para medir el rendimiento inicial de Naive Bayes."
)

Experimento: NB-01 GaussianNB baseline
Train accuracy: 0.6190
5-fold CV accuracy: 0.5390 (+/- 0.0377)
Notas: Baseline sin preprocesamiento. Sirve para medir el rendimiento inicial de Naive Bayes.
